# Police Crime Data Ingestion Notebook

This notebook is used to ingest the raw crime data into a unified Delta file for cleaning and transformation.

#### Imports

In [0]:
import os
from pyspark.sql import functions as F
from functools import reduce

#### Configuration

In [0]:
# Root project directory
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Raw data directory
crime_data_path = os.path.join(
    repo_root,
    "data",
    "raw",
    "crime"
)

# Selected police forces
selected_forces = [
    "greater-manchester",
    "south-wales",
    "sussex",
    "west-midlands"
]

In [0]:
all_files = os.listdir(crime_data_path)
print(f"Total files found: {len(all_files)}")

In [0]:
crime_files = []

for file in all_files:
    # Ensure only street crime CSV files are included
    if file.endswith("-street.csv"):
        # Check if file belongs to selected police forces
        if any(force in file for force in selected_forces):
            crime_files.append(file)

print(f"Selected files: {len(crime_files)}")

# Check if any files were selected
if len(crime_files) == 0:
    raise Exception("No matching police crime files found.")

#### Ingest Files

In [0]:
dataframes = []

for file in crime_files:
    # Full file path
    file_path = os.path.join(crime_data_path, file)
    print(f"Reading: {file}")
    # Read CSV
    df = (spark.read.csv(
        file_path,
        header=True,
        inferSchema=True
    ))

    # Standardise Column Names
    df = (
        df
        .withColumnRenamed("Crime ID", "crime_id")
        .withColumnRenamed("Month", "month")
        .withColumnRenamed("Reported by", "reported_by")
        .withColumnRenamed("Falls within", "falls_within")
        .withColumnRenamed("Longitude", "longitude")
        .withColumnRenamed("Latitude", "latitude")
        .withColumnRenamed("Location", "location")
        .withColumnRenamed("LSOA code", "lsoa_code")
        .withColumnRenamed("LSOA name", "lsoa_name")
        .withColumnRenamed("Crime type", "crime_type")
        .withColumnRenamed("Last outcome category", "last_outcome_category")
        .withColumnRenamed("Context", "context")
    )
    
    # Extract file name and extract relevant information
    filename_parts = file.replace(".csv", "").split("-")
    
    year = filename_parts[0]
    month = filename_parts[1] 
    police_force = "-".join(filename_parts[2:-1])
    reporting_month = f"{year}-{month}"
    df = (
        df
        .withColumn("source_file", F.lit(file))
        .withColumn("police_force", F.lit(police_force))
        .withColumn("reporting_month", F.lit(reporting_month))
    )

    dataframes.append(df)
    print(f"Done Reading: {file}")

In [0]:
# Union all dataframes
raw_crime_df = reduce(
    lambda df1, df2: df1.unionByName(df2),
    dataframes
)

#### Sample of the raw Data

In [0]:
display(raw_crime_df.orderBy(F.rand()).limit(25))

#### Validation Checks

In [0]:
# Total rows
total_rows = raw_crime_df.count()
print(f"Total rows ingested: {total_rows}")

# Distinct police forces
distinct_forces = raw_crime_df.select(
    "police_force"
).distinct().count()
print(f"Distinct police forces: {distinct_forces}")

# Distinct reporting months
distinct_months = raw_crime_df.select(
    "reporting_month"
).distinct().count()
print(f"Distinct reporting months: {distinct_months}")

#### Export for Next Stage

In [0]:
(
    raw_crime_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("raw_police_crime")
)
print("Delta files saved successfully.")